# eALS on Spark — small-data demo

This notebook runs the whole pipeline end to end on a tiny, committed sample so it can be checked in well under a minute, without downloading the real Yelp dataset.

**Dataset**: `data/sample/yelp_sample.rating` — 300 users sampled from the real Yelp data (`data/raw/yelp.rating`), all their interactions kept, ids reindexed densely. Same 4-column tab-separated format as the source file: `user\titem\trating\ttimestamp`. After deduplication: 300 users, 4,955 items, 7,440 interactions.

**What this notebook demonstrates, in order:**
1. Load the sample data
2. Fit eALS sequentially in numpy (Algorithm 1) — the correctness oracle
3. Fit eALS in parallel on Spark (RDD + `mapPartitions`) — the actual deliverable
4. Verify the two agree numerically (proof that Spark's parallelization is exact, not approximate)
5. Compare against the Spark MLlib ALS baseline
6. Run the online/incremental update (Algorithm 2) and the Structured Streaming micro-batch driver

Every code cell states its inputs and outputs in a comment before running.

In [1]:
# Input: none (environment setup only). Output: none — puts src/ on the path and starts a local SparkSession.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np

from eals.spark_utils import build_session

spark = build_session("demo_small_data", n_cores=4)
sc = spark.sparkContext
print("Spark version:", spark.version, "| master:", sc.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 21:56:39 WARN Utils: Your hostname, DESKTOP-APAYAO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/29 21:56:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/29 21:56:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0 | master: local[4]


## 1. Load the sample dataset

In [2]:
# Input: data/sample/yelp_sample.rating (tab-separated user/item/rating/timestamp, 7,774 raw rows).
# Output: a Dataset(users, items, ts) of 7,440 deduplicated interactions, plus a printed stats dict.
from eals import experiment as xp
from eals.data_prep import describe

SAMPLE_PATH = ROOT / "data" / "sample" / "yelp_sample.rating"
data = xp.load_yelp_paper_ratings(SAMPLE_PATH)

print(describe(data.users, data.items))

{'users': 300, 'items': 4955, 'interactions': 7440, 'sparsity_pct': 99.49949545913219, 'avg_per_user': 24.8, 'avg_per_item': 1.501513622603431}


## 2. Correctness oracle — sequential numpy implementation of Algorithm 1

This is the ground truth every Spark result is checked against (see `tests/test_spark_eals.py` for the full proof at 1e-9 tolerance). We use a leave-one-out split (paper's offline protocol, §5.1: each user's latest interaction held out) and the paper's own popularity-aware weighting (Eq. 8).

In [3]:
# Input: data.users/items/ts (the arrays loaded above), K=16, c0=512, alpha=0.4 (paper's stated Yelp optimum).
# Output: an EALSModel (P, Q latent factors) and a per-iteration objective history — should be non-increasing.
from eals import interactions as itx, numpy_reference as ref
from eals import evaluation as ev
from eals.weighting import item_confidence

K, LAM, N_ITERS = 16, 0.01, 15

tr_u, tr_i, te_u, te_i = ev.leave_one_out_split(data.users, data.items, data.ts)
n_users, n_items = data.n_users, data.n_items
train = itx.build(tr_u, tr_i, n_users, n_items)

c = item_confidence(train.item_counts(), c0=512.0, alpha=0.4)

numpy_model = ref.fit(train, K=K, c=c, lam=LAM, n_iters=N_ITERS, seed=0)
objective_history = [h["objective"] for h in numpy_model.history]
print("objective per iteration:", [round(o, 1) for o in objective_history])
print("non-increasing:", all(a >= b - 1e-6 for a, b in zip(objective_history, objective_history[1:])))

objective per iteration: [4409.9, 3725.4, 3617.9, 3564.6, 3531.8, 3509.4, 3493.2, 3480.9, 3471.3, 3463.6, 3457.3, 3452.1, 3447.6, 3443.8, 3440.5]
non-increasing: True


## 3. The same fit, parallelized on Spark (RDD + `mapPartitions`)

This is the actual deliverable: Algorithm 1 parallelized over users (then items) within each phase, broadcasting only the small shared state (`Q`, `S^q`) — see `src/eals/spark_eals.py` and `notes/05_recap.md` §5 for the full design rationale.

In [4]:
# Input: the same (tr_u, tr_i) pairs as above, same K/c/lam/seed — same problem, different execution engine.
# Output: a SparkEALSResult (P, Q) plus a history of per-iteration timings and cache-build times.
from eals import spark_eals

user_rdd, item_rdd = spark_eals.build_views(sc, list(zip(tr_u.tolist(), tr_i.tolist())), n_parts=4)
spark_result = spark_eals.fit(sc, user_rdd, item_rdd, n_users, n_items, K=K, c=c, lam=LAM, n_iters=N_ITERS, seed=0, n_parts=4)

for row in spark_result.history[:3] + spark_result.history[-2:]:
    print(f"iter {row['iter']:2d}  t_iter={row['t_iter']:.3f}s  (S^q={row['t_Sq']:.3f}s user={row['t_user']:.3f}s S^p={row['t_Sp']:.3f}s item={row['t_item']:.3f}s)")

iter  0  t_iter=0.967s  (S^q=0.195s user=0.212s S^p=0.127s item=0.431s)
iter  1  t_iter=0.594s  (S^q=0.096s user=0.149s S^p=0.111s item=0.237s)
iter  2  t_iter=0.296s  (S^q=0.020s user=0.080s S^p=0.064s item=0.133s)
iter 13  t_iter=0.667s  (S^q=0.073s user=0.122s S^p=0.098s item=0.374s)
iter 14  t_iter=0.632s  (S^q=0.054s user=0.134s S^p=0.114s item=0.330s)


## 4. Verify Spark matches the numpy oracle exactly

The same check `tests/test_spark_eals.py` runs automatically, made visible here. If §4.2.1's
argument — or our implementation of it — were wrong, Spark's parallel result wouldn't match
the sequential one.

In [5]:
# Input: numpy_model.P/Q and spark_result.P/Q (both already computed above, same seed/hyperparameters).
# Output: two printed max-abs-difference numbers — should both be ~1e-9 or smaller (float roundoff only).
diff_P = np.max(np.abs(numpy_model.P - spark_result.P))
diff_Q = np.max(np.abs(numpy_model.Q - spark_result.Q))
print(f"max |P_numpy - P_spark| = {diff_P:.2e}")
print(f"max |Q_numpy - Q_spark| = {diff_Q:.2e}")
assert diff_P < 1e-6 and diff_Q < 1e-6, "Spark and numpy diverged — the exact-parallelism claim would be false"
print("OK: Spark's parallel result is numerically identical to the sequential oracle.")

max |P_numpy - P_spark| = 1.29e-14
max |Q_numpy - Q_spark| = 1.50e-15
OK: Spark's parallel result is numerically identical to the sequential oracle.


## 5. Evaluate: eALS vs. the Spark MLlib ALS baseline

HR@10 / NDCG@10 on the held-out (leave-one-out) test interactions, computed distributed (`evaluation.evaluate`, one GEMM per partition-batch rather than one dot product per user).

In [6]:
# Input: spark_result.P/Q (the model just fit) and the held-out (te_u, te_i) pairs from the leave-one-out split.
# Output: a metrics dict (HR@10, NDCG@10, HR@100, NDCG@100, mean_rank) for eALS.
eals_metrics = ev.evaluate(sc, spark_result.P, spark_result.Q, te_u, te_i, top_ks=(10, 100), n_parts=4)
print("eALS:", {k: round(v, 4) for k, v in eals_metrics.items()})

eALS: {'n_cases': 300.0, 'mean_rank': 1997.2267, 'HR@10': 0.01, 'NDCG@10': 0.0038, 'HR@100': 0.0967, 'NDCG@100': 0.0195}


In [7]:
# Input: (tr_u, tr_i) training pairs, same K. Output: an MLlib ALS model's factors + the same metrics dict, for comparison.
from eals import baseline_als

mllib_out = baseline_als.fit(spark, tr_u, tr_i, n_users, n_items, K=K, max_iter=N_ITERS, n_parts=4)
mllib_metrics = ev.evaluate(sc, mllib_out["P"], mllib_out["Q"], te_u, te_i, top_ks=(10, 100), n_parts=4)
print("MLlib ALS:", {k: round(v, 4) for k, v in mllib_metrics.items()})
print(f"\nNote: this tiny 300-user sample is for demonstrating the pipeline works end to end, not for\n"
      f"drawing accuracy conclusions — see results/*.csv and notes/02-04_*.md for the real-scale experiments.")

MLlib ALS: {'n_cases': 300.0, 'mean_rank': 1996.0933, 'HR@10': 0.0267, 'NDCG@10': 0.0101, 'HR@100': 0.1167, 'NDCG@100': 0.0268}

Note: this tiny 300-user sample is for demonstrating the pipeline works end to end, not for
drawing accuracy conclusions — see results/*.csv and notes/02-04_*.md for the real-scale experiments.


## 6. Online/incremental update — Algorithm 2

Two things demonstrated here: (a) a single incremental update on a held-out interaction, and (b) the Structured Streaming micro-batch driver that feeds Algorithm 2 from a chunked file source (`foreachBatch`, one file per micro-batch) — see `src/eals/spark_eals_online.py`.

In [8]:
# Input: the offline-trained (spark_result.P, spark_result.Q, c) and one held-out (user, item) pair.
# Output: the item's rank before vs. after one incremental update — demonstrates Algorithm 2's O(K^2 + |R_u|K) refit.
from eals import spark_eals_online as online

state = online.OnlineState.from_offline_fit(
    spark_result.P, spark_result.Q, c, lam=LAM, w_new=4.0, train_users=tr_u, train_items=tr_i
)

u, i = int(te_u[0]), int(te_i[0])
rank_before = ev.rank_of_truth(state.scores_for(u), i, None)
state.apply_interaction(u, i, n_iters=1)
rank_after = ev.rank_of_truth(state.scores_for(u), i, None)
print(f"user {u}, item {i}: rank before update = {rank_before}, rank after = {rank_after} (out of {n_items} items)")

user 0, item 333: rank before update = 3420, rank after = 17 (out of 4955 items)


In [9]:
# Input: the remaining held-out interactions (te_u[1:], te_i[1:]), written as 3 small JSON files (micro-batches).
# Output: one result dict per streamed event (batch_id, user, item, prior_count, rank) — printed for inspection.
STREAM_DIR = ROOT / "notebooks" / "_demo_stream"
CKPT_DIR = ROOT / "notebooks" / "_demo_ckpt"

n_batches = online.write_stream_batches(te_u[1:11], te_i[1:11], data.ts[:10], STREAM_DIR, rows_per_batch=3)
print(f"wrote {n_batches} micro-batch files to {STREAM_DIR}")

stream_results = online.run_streaming(spark, state, STREAM_DIR, CKPT_DIR, n_iters_online=1)
for r in stream_results:
    print(r)

wrote 4 micro-batch files to /mnt/d/1 Dauphine/1 M2/MLBD/MFOR/notebooks/_demo_stream


{'batch_id': 0, 'user': 1, 'item': 147, 'ts': 1136736000, 'prior_count': 81, 'rank': 168}
{'batch_id': 0, 'user': 2, 'item': 572, 'ts': 1136736000, 'prior_count': 15, 'rank': 3274}
{'batch_id': 0, 'user': 3, 'item': 25, 'ts': 1136736000, 'prior_count': 45, 'rank': 5}
{'batch_id': 1, 'user': 4, 'item': 324, 'ts': 1136736000, 'prior_count': 24, 'rank': 3326}
{'batch_id': 1, 'user': 5, 'item': 165, 'ts': 1137427200, 'prior_count': 50, 'rank': 2012}
{'batch_id': 1, 'user': 6, 'item': 419, 'ts': 1137427200, 'prior_count': 13, 'rank': 2563}
{'batch_id': 2, 'user': 7, 'item': 703, 'ts': 1152633600, 'prior_count': 23, 'rank': 172}
{'batch_id': 2, 'user': 8, 'item': 699, 'ts': 1152633600, 'prior_count': 9, 'rank': 1977}
{'batch_id': 2, 'user': 9, 'item': 1190, 'ts': 1166630400, 'prior_count': 79, 'rank': 1870}
{'batch_id': 3, 'user': 10, 'item': 1737, 'ts': 1167667200, 'prior_count': 18, 'rank': 3285}


## 7. Summary

This notebook ran, on a tiny committed sample: the numpy correctness oracle, the Spark RDD implementation of Algorithm 1, a numeric proof that the two agree exactly, an MLlib ALS baseline comparison, and both the single-update and Structured-Streaming-micro-batch paths of Algorithm 2 — the full pipeline, top to bottom.

For the real-scale results (Yelp's full 731,671 interactions) behind every claim in the report — K-scalability up to 1024, parallelism speedup, the weighting sweep, and the full online-protocol experiments — see `results/*.csv` and the corresponding `notes/*.md` files.

In [10]:
# Input: none. Output: none — releases the Spark session and removes the small scratch files this notebook wrote.
import shutil

shutil.rmtree(STREAM_DIR, ignore_errors=True)
shutil.rmtree(CKPT_DIR, ignore_errors=True)
spark.stop()